### Exploratory macro-driver relationships

Compare monthly changes in log demand with changes in lagged macro indicators by product family. These are exploratory associations, not causal effects; they are used to prioritize candidate drivers and do not replace temporal backtesting.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from src.feature_engineering import MACRO_COLUMNS

demand = pd.read_csv(
    PROJECT_ROOT / "outputs" / "demand_reconstruction" / "train_with_reconstructed_demand.csv",
    low_memory=False,
)
macro = pd.read_csv(
    PROJECT_ROOT / "outputs" / "cleaned" / "macro_indicators_cleaned.csv",
    low_memory=False,
)
demand["Unconstrained_Demand"] = pd.to_numeric(demand["Unconstrained_Demand"], errors="coerce")
demand = demand.loc[
    demand["Date"].between("2019-01", "2026-10")
    & demand["Product_Family"].notna()
    & demand["Unconstrained_Demand"].ge(0)
].copy()
demand["Date_Period"] = pd.PeriodIndex(demand["Date"], freq="M")
family_month = demand.groupby(["Product_Family", "Date_Period"])[
    "Unconstrained_Demand"
].sum(min_count=1)
macro["Date_Period"] = pd.PeriodIndex(macro["Date"], freq="M")
for column in MACRO_COLUMNS:
    macro[column] = pd.to_numeric(macro[column], errors="coerce")
macro_month = macro.groupby("Date_Period")[list(MACRO_COLUMNS)].mean()
calendar = pd.period_range("2019-01", "2026-10", freq="M")

relationship_rows = []
for family in family_month.index.get_level_values("Product_Family").unique():
    target = family_month.xs(family, level="Product_Family").reindex(calendar)
    target_change = np.log1p(target.clip(lower=0)).diff()
    for indicator in MACRO_COLUMNS:
        driver = macro_month[indicator].reindex(calendar)
        for lag in (0, 1, 3):
            driver_change = driver.shift(lag).diff()
            paired = pd.concat(
                [target_change.rename("Demand_Change"), driver_change.rename("Driver_Change")],
                axis=1,
            ).dropna()
            relationship_rows.append({
                "Product_Family": family,
                "Macro_Indicator": indicator,
                "Macro_Lag_Months": lag,
                "Paired_Months": len(paired),
                "Pearson_Correlation": paired["Demand_Change"].corr(paired["Driver_Change"]),
                "Spearman_Correlation": paired["Demand_Change"].corr(
                    paired["Driver_Change"], method="spearman"
                ),
            })

macro_relationships = pd.DataFrame(relationship_rows)
relationship_path = PROJECT_ROOT / "outputs" / "features" / "macro_relationships_by_family.csv"
macro_relationships.to_csv(relationship_path, index=False, encoding="utf-8-sig")
display(
    macro_relationships.assign(
        Absolute_Pearson=macro_relationships["Pearson_Correlation"].abs()
    ).sort_values("Absolute_Pearson", ascending=False).head(20)
)
assert len(macro_relationships) == demand["Product_Family"].nunique() * len(MACRO_COLUMNS) * 3
print(f"Saved exploratory correlations to {relationship_path}")

# 04 - Features and external drivers

Build a complete SKU-region month grid so lags mean calendar months rather than previous available rows. Missing months and targets remain missing, never zero-filled. Features include calendar seasonality, lag/rolling demand, intermittency, product lifecycle, known event scope counts, and lagged macro indicators.

Leakage controls: historical rolling features shift demand by one month; macro features use 1- and 3-month lags; test features use history through Oct 2026 and the known test metadata. No same-month realized macro values are used.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "outputs" / "demand_exploration").is_dir():
            return candidate
    raise FileNotFoundError("Run notebooks 01-03 before feature engineering.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
from src.feature_engineering import run_feature_engineering

metadata = run_feature_engineering(PROJECT_ROOT)
print(f"Training feature rows: {metadata['train_feature_rows']:,}")
print(f"Test feature rows: {metadata['test_feature_rows']:,}")
print(f"History rows excluded for incomplete keys/out-of-window dates: {metadata['dropped_history_rows_missing_key_or_outside_window']:,}")
print("Feature leakage controls:")
for rule in metadata["leakage_controls"]:
    print(f"- {rule}")

In [ ]:
import pandas as pd
from IPython.display import display

feature_dir = PROJECT_ROOT / "outputs" / "features"
train_features = pd.read_csv(feature_dir / "train_features.csv", low_memory=False)
test_features = pd.read_csv(feature_dir / "test_features.csv", low_memory=False)

important = [
    "Date", "SKU_ID", "Region", "Unconstrained_Demand", "lag_1", "lag_3",
    "rolling_mean_3", "zero_share_prior_12", "Month_Sin", "Known_Scoped_Event_Count",
    "ISM_Manufacturing_PMI_lag_1", "Months_Since_Launch", "Months_To_EOL",
]
display(train_features[[column for column in important if column in train_features]].head())
display(test_features[[column for column in important if column in test_features]].head())

assert len(test_features) == 58_219
assert not test_features.duplicated(["Date", "SKU_ID", "Region"]).any()
assert "ISM_Manufacturing_PMI" not in test_features.columns
assert {"lag_1", "lag_3", "rolling_mean_3", "Known_Scoped_Event_Count"}.issubset(test_features.columns)
print("Validation passed: one test feature row per test key and no same-month macro inputs.")

The feature outputs and column/availability documentation are in `outputs/features/`. Use `train_features.csv` for time-based evaluation and `test_features.csv` for forecast keys. For horizons beyond one month, do not treat future target-dependent lags as known; use recursive prediction or origin-safe features in the modeling notebook.